In [1]:
import numpy as np
import napari
from morphotrack import networks, analysis, utils, meshes
import torch
import trimesh
from tqdm import tqdm
import os
import pandas as pd

/home/tmurakami/src/morphotrack/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set input
fix_mesh = trimesh.load("../data/mesh_pia.ply")
mov_mesh = trimesh.load("../data/mesh_wm.ply")
checkpoint_path = '../data/vhat_field_checkpoint.pth'
centroid_path = '../data/rRNA.csv'

fix_vertices = fix_mesh.vertices
mov_vertices = mov_mesh.vertices
fix_face = fix_mesh.faces
mov_face = mov_mesh.faces

### Parameter settings
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fix_original = torch.from_numpy(np.asarray(fix_vertices)).to(device, dtype=torch.float32)
mov_original = torch.from_numpy(np.asarray(mov_vertices)).to(device, dtype=torch.float32)

### Regular option
v_field = networks.SimpleMLP(hidden_sizes=[256, 128, 64], use_norm=True, use_residual=True, activation_func='SiLU').to(device)
v_field_ckpt = torch.load(checkpoint_path, map_location='cuda')
v_field.load_state_dict(v_field_ckpt['model_state_dict'])

v_field.eval()  # Set to evaluation mode

### bbox bounds for normalization (loaded from checkpoint to match training)
pos_min = v_field_ckpt['pos_min'].to(device, dtype=torch.float32)
pos_max = v_field_ckpt['pos_max'].to(device, dtype=torch.float32)
pos_min_np = pos_min.cpu().numpy()
pos_max_np = pos_max.cpu().numpy()


df = pd.read_csv(centroid_path)
centroid_cols = ['centroid-0', 'centroid-1', 'centroid-2']
centroids  = (df[centroid_cols]).values
sel = np.all(np.logical_and(centroids >= pos_min_np, centroids <= pos_max_np),axis=1)
centroids = centroids[sel]

### Load Model

In [3]:
v_field_norm = networks.NormalizedField(v_field, pos_min, pos_max)

v_field_norm.eval()
thickness = 2300
steps = 50
dt = thickness/steps

with torch.no_grad():
    mov_traj = analysis.integrate_rk4(mov_original, v_field_norm, steps=steps, dt=dt)

In [ ]:
def _batched_first_intersection(trajs_np, target_mesh, cum_s, seg_lens, abs_dt):
    steps = trajs_np.shape[0] - 1
    N     = trajs_np.shape[1]

    starts = trajs_np[:-1].reshape(-1, 3)
    ends   = trajs_np[1:].reshape(-1, 3)
    dirs   = ends - starts
    lens   = np.linalg.norm(dirs, axis=1)

    valid     = lens > 1e-8
    valid_idx = np.where(valid)[0]
    v_starts  = starts[valid]
    v_lens    = lens[valid]
    v_dirs    = dirs[valid] / v_lens[:, None]

    locations, ray_idx, tri_idx = target_mesh.ray.intersects_location(
        ray_origins=v_starts, ray_directions=v_dirs, multiple_hits=False,
    )

    xyz = np.full((N, 3), np.nan, dtype=np.float32)
    s   = np.full(N,      np.nan, dtype=np.float32)
    t   = np.full(N,      np.nan, dtype=np.float32)
    fid = np.full(N,      -1,     dtype=np.int64)              # ← new
    hit = np.zeros(N, dtype=bool)
    if len(locations) == 0:
        return xyz, s, t, hit, fid

    hit_dist = np.linalg.norm(locations - v_starts[ray_idx], axis=1)
    in_seg   = hit_dist <= v_lens[ray_idx]
    if not in_seg.any():
        return xyz, s, t, hit, fid

    flat_seg  = valid_idx[ray_idx[in_seg]]
    step_idx  = flat_seg // N
    traj_idx  = flat_seg %  N
    alpha     = hit_dist[in_seg] / v_lens[ray_idx[in_seg]]
    locs      = locations[in_seg]
    tri_chosen = tri_idx[in_seg]                                # ← keep this

    order     = np.lexsort((step_idx, traj_idx))
    first     = np.empty(order.size, dtype=bool)
    first[0]  = True
    first[1:] = traj_idx[order][1:] != traj_idx[order][:-1]
    chosen    = order[first]

    ti, si, ai = traj_idx[chosen], step_idx[chosen], alpha[chosen]
    li, fi     = locs[chosen], tri_chosen[chosen]
    xyz[ti] = li
    s[ti]   = cum_s[si, ti] + ai * seg_lens[si, ti]
    t[ti]   = (si + ai) * abs_dt
    fid[ti] = fi                                                # ← face id
    hit[ti] = True
    return xyz, s, t, hit, fid


def _integrate_and_hit(X0_all, target_mesh, dt_signed, steps, chunk, label):
    N        = X0_all.shape[0]
    trajs_np = np.empty((steps + 1, N, 3), dtype=np.float32)
    v_field_norm.eval()
    with torch.no_grad():
        for a in tqdm(range(0, N, chunk), desc=f"integrate {label}"):
            b  = min(a + chunk, N)
            tc = analysis.integrate_rk4(
                X0_all[a:b].clone(), v_field_norm, steps=steps, dt=dt_signed,
            )
            trajs_np[:, a:b, :] = tc.cpu().numpy()
            del tc; torch.cuda.empty_cache()

    seg_lens = np.linalg.norm(np.diff(trajs_np, axis=0), axis=2)
    cum_s    = np.concatenate([np.zeros((1, N), dtype=seg_lens.dtype),
                               np.cumsum(seg_lens, axis=0)], axis=0)

    return _batched_first_intersection(
        trajs_np, target_mesh, cum_s, seg_lens, abs_dt=abs(dt_signed),
    )


# ---- Run -----------------------------------------------------------------
X0_all  = torch.as_tensor(centroids, dtype=torch.float32, device=device)
N       = X0_all.shape[0]
chunk   = 4096

xyz_mov, s_mov, t_mov, hit_mov, fid_mov = _integrate_and_hit(
    X0_all, mov_mesh, -dt, steps, chunk, "mov (backward)")
xyz_fix, s_fix, t_fix, hit_fix, fid_fix = _integrate_and_hit(
    X0_all, fix_mesh, +dt, steps, chunk, "fix (forward)")

# ---- Combined per-point statistics --------------------------------------
valid   = hit_mov & hit_fix
total_s = s_mov + s_fix
total_t = t_mov + t_fix
frac_s  = s_mov / total_s
frac_t  = t_mov / total_t

print(f"hit mov:  {hit_mov.sum():>7} / {N}")
print(f"hit fix:  {hit_fix.sum():>7} / {N}")
print(f"valid:    {valid.sum():>7} / {N}    "
      f"median total s = {np.nanmedian(total_s[valid]):.1f},  "
      f"median total t = {np.nanmedian(total_t[valid]):.1f}")

xyz_mov_valid = xyz_mov[valid]
fid_mov_valid = fid_mov[valid]
xyz_fix_valid = xyz_fix[valid]
total_s_valid = total_s[valid]
s_mov_valid = s_mov[valid]
s = s_mov_valid / total_s_valid

integrate mov (backward): 100%|██████████| 945/945 [01:30<00:00, 10.49it/s]


### UVS transformation

In [ ]:
# choose id to determine the orientation.
viewer = napari.Viewer(ndisplay=3)
viewer.add_points(mov_vertices, size=60, face_color="magenta")
viewer.add_points(fix_vertices, size=60, face_color="lime")

In [ ]:
### If just want a uniform embedding without c0
align_id = (299,289)

values = np.ones(mov_vertices.shape[0])
embedding = meshes.embed_mesh_isomap(mov_vertices, mov_face, values, align_vertices=align_id, align_direction=(0, 1), flip_v=False) # (1836,1840)
mesh_mapper = meshes.MeshMapper(embedding, mov_vertices, mov_face)
mov_vertices_uv = mesh_mapper.xyz_to_uv(mov_vertices)

In [ ]:
# get uv of the points.
uv_mov_valid = mesh_mapper.xyz_to_uv(xyz_mov_valid)

In [ ]:
from matplotlib.tri import Triangulation

# ---- inputs --------------------------------------------------------------
# uv_mov_valid : (M, 2)   per-point UV
# s            : (M,)     per-point s ∈ [0, 1]
# mov_mesh.faces, mov_vertices_uv
faces    = np.asarray(mov_mesh.faces,    dtype=np.int64)
verts_uv = np.asarray(mov_vertices_uv,   dtype=np.float64)
Nv, Nf   = verts_uv.shape[0], faces.shape[0]
s_grid_num = 1000

# ---- 1. Make winding CCW for matplotlib ---------------------------------
def signed_uv_area(F):
    v0, v1, v2 = (verts_uv[F[:, k]] for k in range(3))
    return 0.5 * ((v1[:, 0] - v0[:, 0]) * (v2[:, 1] - v0[:, 1])
                - (v2[:, 0] - v0[:, 0]) * (v1[:, 1] - v0[:, 1]))

sa = signed_uv_area(faces)
if (sa < 0).sum() > (sa > 0).sum():
    faces = faces[:, ::-1]                       # flip whole mesh
sa  = signed_uv_area(faces)
bad = sa <= 0
print(f"masking {int(bad.sum())} / {Nf} mis-wound faces")

# ---- 2. Trifinder + exact per-point face id ----------------------------
tri_obj = Triangulation(verts_uv[:, 0], verts_uv[:, 1], triangles=faces)
tri_obj.set_mask(bad)
fid_pt  = tri_obj.get_trifinder()(uv_mov_valid[:, 0], uv_mov_valid[:, 1])
valid   = fid_pt >= 0
print(f"unassigned: {int((~valid).sum())} / {len(uv_mov_valid)}")

# ---- 3. Barycentric weights for the valid points -----------------------
fi  = fid_pt[valid]
p   = uv_mov_valid[valid]
v0, v1, v2 = (verts_uv[faces[fi, k]] for k in range(3))
e1, e2, ep = v1 - v0, v2 - v0, p - v0
d00, d01, d11 = (e1 * e1).sum(1), (e1 * e2).sum(1), (e2 * e2).sum(1)
d20, d21      = (ep * e1).sum(1), (ep * e2).sum(1)
denom = d00 * d11 - d01 * d01
w1    = (d11 * d20 - d01 * d21) / denom
w2    = (d00 * d21 - d01 * d20) / denom
w0    = 1.0 - w1 - w2
weights = np.stack([w0, w1, w2], axis=1)            # (M_valid, 3)

# ---- 4. Per-vertex (s_grid+1, Nv) histogram ---------------------------
s_bins = np.linspace(0, 1, s_grid_num + 1)   # shape (s_grid_num + 1,)
bin_idx = np.clip((np.clip(s, 0, 1) * s_grid_num).astype(np.int64), 0, s_grid_num)
v_ids   = faces[fi]                                  # (M_valid, 3)

M = np.zeros((s_grid_num + 1, Nv), dtype=np.float64)
for k in range(3):
    np.add.at(M, (bin_idx[valid], v_ids[:, k]), weights[:, k])

print(f"binned {int(valid.sum())} points → M shape {M.shape}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 8))
ax.triplot(tri_obj, color="k", lw=0.2)
ax.set_aspect("equal")
plt.show()

In [ ]:
viewer = napari.Viewer()
viewer.add_image(M)

In [ ]:
mesh_idx = np.arange(mov_vertices.shape[0])
np.savez_compressed(os.path.join('./uvt_value_rrna.npz'),
    intensities = M,        # (T, N)  the actual values
    t_grid      = s_bins,             # (T,)    so 06 knows the t axis
    uv          = mov_vertices_uv,    # (N, 2)  per-vertex (u,v)
    xyz         = mov_vertices,       # (N, 3)
    faces       = mov_face,
    mesh_idx    = mesh_idx,           # (N,)    indices into mov_vertice
)

import pickle
# Save
with open(os.path.join('./mapper_rrna.npz'), "wb") as f:
    pickle.dump(mesh_mapper, f)

In [ ]:
np.savez_compressed(os.path.join('./cells_rrna.npz'),
    xyz = centroids[hit_mov & hit_fix],
    uv = uv_mov_valid,                 # (N, 2)  per-vertex (u,v)
    s = s,           # (N,)    indices into mov_vertice
    L = total_s_valid,
    fid_mov = fid_mov_valid,
    xyz_on_mov = xyz_mov_valid,
)

# # Load (in a later session/notebook)
# with open("./05_output/mesh_mapper.pkl", "rb") as f:
#     mapper = pickle.load(f)
